In [ ]:
# ==========================================
# 1. Team Configuration
# ==========================================
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

project_root = Path(r"C:\Users\DELL\dsba6171-microcosm-starter")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

load_dotenv(project_root / ".env", override=True)

# ==========================================
# 2. Domain Team Structured Data
# ==========================================
csv_path = project_root / "data" / "structured" / "raw" / "synthetic_knowledge_items.csv"
chroma_dir = project_root / "data" / "processed" / "chroma"

# ==========================================
# 3. Validate Schema
# ==========================================
df = pd.read_csv(csv_path)
required_cols = {"ki_id", "ki_topic", "ki_text", "jurisdiction", "authority_status", "effective_date"}
assert required_cols.issubset(set(df.columns)), f"Schema missing required fields: {required_cols - set(df.columns)}"
print("Schema Validation Passed. Found columns:", list(df.columns))

# ==========================================
# 4. Apply/Test Semantic Expectations
# ==========================================
# Verify dataset contains intentional ambiguity for baseline vs filtered testing
print(f"Loaded {len(df)} domain records across jurisdictions: {df['jurisdiction'].unique().tolist()}")

# ==========================================
# 5. Create Canonical AI-Ready Representation
# ==========================================
from langchain_community.document_loaders import CSVLoader

# ==========================================
# 6. Load Team Knowledge Documents
# ==========================================
# ==========================================
# 7. Create Chunks + Preserve Metadata
# ==========================================
loader = CSVLoader(
    file_path=str(csv_path),
    metadata_columns=["ki_id", "jurisdiction", "authority_status", "effective_date"]
)
documents = loader.load()
print(f"Created {len(documents)} document chunks with preserved metadata.")

# ==========================================
# 8. Embed + Load into Chroma
# ==========================================
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=str(chroma_dir)
)

# ==========================================
# 9. Run Baseline Semantic Retrieval
# ==========================================
query = "What is the policy for ChromaDB vector database storage?"
baseline_results = vectorstore.similarity_search(query, k=2)

print("\n--- BASELINE SEMANTIC RETRIEVAL ---")
for i, doc in enumerate(baseline_results, 1):
    print(f"[{i}] ID: {doc.metadata.get('ki_id')} | Status: {doc.metadata.get('authority_status')} | Content: {doc.page_content[:80]}...")

# ==========================================
# 10. Run Metadata/Context-Filtered Retrieval
# ==========================================
filtered_results = vectorstore.similarity_search(
    query,
    k=2,
    filter={"$and": [{"jurisdiction": "US"}, {"authority_status": "active"}]}
)

print("\n--- CONTEXT-FILTERED RETRIEVAL (US + Active) ---")
for i, doc in enumerate(filtered_results, 1):
    print(f"[{i}] ID: {doc.metadata.get('ki_id')} | Status: {doc.metadata.get('authority_status')} | Content: {doc.page_content[:80]}...")

# ==========================================
# 11. Compare the Results
# ==========================================
target_ki_id = "KI-001"
baseline_hit = any(doc.metadata.get("ki_id") == target_ki_id for doc in baseline_results)
filtered_hit = any(doc.metadata.get("ki_id") == target_ki_id for doc in filtered_results)

print("\n--- COMPARISON (Hit@1 Assessment) ---")
print(f"Target '{target_ki_id}' in Baseline Top-1: {baseline_results[0].metadata.get('ki_id') == target_ki_id}")
print(f"Target '{target_ki_id}' in Filtered Top-1: {filtered_results[0].metadata.get('ki_id') == target_ki_id}")

# ==========================================
# 12. Trace Retrieved Evidence to Source
# ==========================================
top_doc = filtered_results[0]
print("\n--- SOURCE TRACE ---")
print(f"Selected KI ID: {top_doc.metadata.get('ki_id')}")
print(f"Source File: {top_doc.metadata.get('source')}")
print(f"Row Index: {top_doc.metadata.get('row')}")
print(f"Effective Date: {top_doc.metadata.get('effective_date')}")

# ==========================================
# 13. Save Test Evidence
# ==========================================
evidence_df = pd.DataFrame([{
    "query": query,
    "top_baseline_id": baseline_results[0].metadata.get("ki_id"),
    "top_filtered_id": filtered_results[0].metadata.get("ki_id"),
    "hit_at_1": filtered_results[0].metadata.get("ki_id") == target_ki_id
}])
evidence_path = project_root / "deliverables" / "retrieval_test_evidence.csv"
evidence_path.parent.mkdir(parents=True, exist_ok=True)
evidence_df.to_csv(evidence_path, index=False)
print(f"\nTest evidence successfully saved to: {evidence_path}")

# ==========================================
# 14. Optional: Generate Grounded AI Response
# ==========================================
from src.vectorstore.chain import get_rag_chain

chain = get_rag_chain()
response = chain.invoke(query)
print("\n--- OPTIONAL GROUNDED AI RESPONSE ---")
print(response)